In [10]:
import json
import shutil
from pathlib import Path
import pandas as pd

# paths
BASE = Path("/kaggle/input/competitions/herbarium-2022-fgvc9")
TRAIN_IMG_DIR = BASE / "train_images"
TRAIN_JSON = BASE / "train_metadata.json"

OUT_ROOT = Path("/kaggle/working")
DEST = OUT_ROOT / "herbarium_topk_by_class"

TOP_K = 50

In [11]:
if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True, exist_ok=True)

# load metadata
with open(TRAIN_JSON) as f:
    train_meta = json.load(f)

images_df = pd.DataFrame(train_meta["images"])
ann_df = pd.DataFrame(train_meta["annotations"])

df = images_df.merge(
    ann_df[["image_id", "category_id"]],
    on="image_id",
    how="inner"
)

# find top-K classes by number of images
counts = df["category_id"].value_counts()
top_classes = counts.head(TOP_K).index.tolist()

print("Top classes selected:")
print(counts.head(TOP_K))

# filter rows
df_top = df[df["category_id"].isin(top_classes)].copy()

Top classes selected:
category_id
19       80
15390    80
1727     80
1726     80
1861     80
9375     80
9309     80
9298     80
9292     80
9621     80
9602     80
9597     80
2505     80
2366     80
2256     80
9123     80
8985     80
8973     80
8972     80
8970     80
9819     80
9744     80
1443     80
2774     80
2750     80
1463     80
9723     80
573      80
572      80
8865     80
10029    80
9838     80
8864     80
8862     80
8861     80
8854     80
579      80
578      80
8800     80
8794     80
8677     80
8435     80
10161    80
13471    80
3046     80
3026     80
2889     80
2876     80
13957    80
10076    80
Name: count, dtype: int64


In [12]:
# copy images into class folders
copied = 0
for _, row in df_top.iterrows():
    file_name = row["file_name"]
    class_name = str(row["category_id"])   # folder name = original category_id

    src = TRAIN_IMG_DIR / file_name
    class_dir = DEST / class_name
    class_dir.mkdir(parents=True, exist_ok=True)

    dst = class_dir / Path(file_name).name
    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} images into {DEST}")

Copied 4000 images into /kaggle/working/herbarium_topk_by_class


In [13]:
zip_base = "/kaggle/working/herbarium_topk_by_class"
archive_path = shutil.make_archive(zip_base, "zip", root_dir=DEST)

print("Created zip:", archive_path)

Created zip: /kaggle/working/herbarium_topk_by_class.zip
